# P04B — Eksplorasi dan pemodelan algoritma

Jalankan P04A terlebih dahulu. Notebook ini memperlihatkan bagaimana beberapa keluarga algoritma digunakan pada data sintesis Bab 4. Hasil latihan tidak menggambarkan kinerja model lapangan. Evaluasi metrik dan tuning sistematis diperdalam pada Bab 5–6.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

DATA = Path("data_bab4")
print("Folder data:", DATA.resolve())


Folder data: C:\DATA\Ryandra Second Brain\30 - PROJECT\Buku Ajar ML\Bab4_Paket\data_bab4


## 1. Periksa skema dan sasaran
Pastikan identitas dan label simulasi tidak masuk sebagai prediktor. Pada contoh jembatan, label memang dibuat dengan aturan sintesis sehingga pemodelan hanya mengajarkan alur supervised learning.

In [2]:
beton = pd.read_csv(DATA / "data_sintesis_kuat_tekan.csv")
jembatan = pd.read_csv(DATA / "data_sintesis_prioritas_jembatan.csv")
perkerasan = pd.read_csv(DATA / "data_sintesis_perkerasan.csv")
sensor = pd.read_csv(DATA / "data_sintesis_sensor.csv")
for nama, df in [("beton", beton),("jembatan",jembatan),("perkerasan",perkerasan),("sensor",sensor)]:
    print(nama, df.shape, "missing:", int(df.isna().sum().sum()))
print("Kelas label sintesis jembatan:")
print(jembatan["prioritas_sintesis"].value_counts().to_dict())


beton (1200, 10) missing: 0
jembatan (600, 7) missing: 0
perkerasan (800, 6) missing: 0
sensor (700, 4) missing: 0
Kelas label sintesis jembatan:
{'prioritas': 308, 'rutin': 292}


## 2. Klasifikasi label latihan jembatan
Label sintesis dibuat dari ambang pada skor simulasi yang menggabungkan lima prediktor. Model mempelajari label tersebut; label bukan kategori inspeksi resmi. ID aset dikeluarkan dari fitur.


In [3]:
bridge_features = ["umur_tahun", "lalin_berat_harian", "risiko_gerusan", "indeks_deteriorasi", "paparan_banjir"]
Xb = jembatan[bridge_features]; yb = jembatan["prioritas_sintesis"]
Xb_train, Xb_test, yb_train, yb_test = train_test_split(Xb, yb, test_size=0.25, random_state=41804, stratify=yb)
classifiers = {"naive_bayes": GaussianNB(), "pohon": DecisionTreeClassifier(max_depth=4, min_samples_leaf=8, random_state=41804)}
for name, clf in classifiers.items():
    clf.fit(Xb_train, yb_train)
    print(name, "contoh kelas:", clf.predict(Xb_test.iloc[:5]).tolist())


naive_bayes contoh kelas: ['prioritas', 'prioritas', 'rutin', 'prioritas', 'prioritas']
pohon contoh kelas: ['prioritas', 'prioritas', 'rutin', 'prioritas', 'prioritas']


## 3. Bandingkan bentuk beberapa regressor
Semua kandidat menerima fitur dan pembagian pelatihan yang sama. Angka prediksi contoh ditampilkan untuk membandingkan bentuk keluaran. Bagian ini tidak menetapkan metode terbaik dan tidak menggantikan evaluasi pada Bab 5.

In [4]:
features = [c for c in beton.columns if c not in ["id_spesimen", "kuat_tekan_MPa"]]
X = beton[features]
y = beton["kuat_tekan_MPa"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=41804)
models = {"linear": LinearRegression(), "pohon": DecisionTreeRegressor(max_depth=5, random_state=41804), "random_forest": RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=41804, n_jobs=-1)}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "contoh prediksi:", model.predict(X_test.iloc[:5]).round(2).tolist())


linear contoh prediksi: [31.98, 23.64, 37.55, 37.25, 31.41]
pohon contoh prediksi: [35.5, 22.1, 32.06, 41.16, 28.54]
random_forest contoh prediksi: [34.57, 25.45, 35.43, 39.74, 28.24]


## 4. Clustering dan PCA
Buang ID dan label pembangkit sebelum membentuk kelompok. Standardisasi dilakukan sebelum menghitung jarak. Komponen PCA merangkum variasi, bukan peringkat kepentingan rekayasa.

In [5]:
cols = ["retak_persen", "kedalaman_alur_mm", "IRI_m_km", "lalu_lintas_tahunan"]
Z = StandardScaler().fit_transform(perkerasan[cols])
clusters = KMeans(n_clusters=4, n_init=10, random_state=41804).fit_predict(Z)
coords = PCA(n_components=2, random_state=41804).fit_transform(Z)
print("Jumlah anggota tiap cluster:", pd.Series(clusters).value_counts().sort_index().to_dict())
print("PCA dua komponen; proporsi variasi:", PCA(n_components=2).fit(Z).explained_variance_ratio_.round(3).tolist())


Jumlah anggota tiap cluster: {0: 198, 1: 200, 2: 201, 3: 201}
PCA dua komponen; proporsi variasi: [0.574, 0.318]


## 5. Menyaring anomali untuk pemeriksaan
Model anomali tidak mendiagnosis kerusakan. Bandingkan baris bertanda oleh generator dengan hasil Isolation Forest hanya sebagai latihan, lalu telaah konteks dan catatan sensor.

In [6]:
sensor["flag_model"] = IsolationForest(contamination=0.03, random_state=41804).fit_predict(sensor[["suhu_C", "getaran_mm_s"]]) == -1
print("Flag generator:", int(sensor["flag_pembangkit"].sum()))
print("Flag Isolation Forest:", int(sensor["flag_model"].sum()))
print(sensor.loc[sensor["flag_model"], ["waktu", "suhu_C", "getaran_mm_s", "flag_pembangkit"]].head(8).to_string(index=False))


Flag generator: 13
Flag Isolation Forest: 21
              waktu  suhu_C  getaran_mm_s  flag_pembangkit
2024-01-01 21:00:00   16.29         1.521             True
2024-01-04 01:00:00   21.53         1.929             True
2024-01-05 19:00:00   14.31         0.425            False
2024-01-08 06:00:00   25.48         0.712            False
2024-01-08 10:00:00   22.58         1.628             True
2024-01-09 01:00:00   21.09         1.684             True
2024-01-10 22:00:00   17.93         1.849             True
2024-01-12 04:00:00   24.44         1.306             True


## 6. Batas penggunaan
Semua hasil bergantung pada seed, rentang dan aturan generator. Jangan mengutip hasil ini sebagai kinerja terhadap beton, jembatan, jalan atau sensor nyata. Simpan pemilihan fitur, target, seed dan pembagian data agar latihan dapat diulang.